# 01G - YOLO11s BDD100K: more data with guaranteed class coverage

This notebook continues from the completed YOLO11s model without changing its
executed notebook or weights. It expands training coverage, guarantees minimum
scene counts for every class, uses efficient 576 px bulk training, retains a
704 px small-object refinement stage, and rejects the new model if it fails to
beat the completed run on the same held-out validation manifest.

**Protected baseline from the completed 01 execution**

| split | precision | recall | F1 | mAP50 | mAP50:95 |
|---|---:|---:|---:|---:|---:|
| validation | 0.609 | 0.503 | 0.551 | 0.526 | 0.286 |
| test | 0.638 | 0.492 | 0.556 | 0.528 | 0.286 |

That result is already archived byte-for-byte. Its selected checkpoint is the
starting point and a candidate in final model selection, so this experiment can
improve it but cannot silently replace it with a worse model.

Object detection does not have one classification-style "accuracy." This
notebook reports precision, recall, F1 at IoU 0.50, mAP50, and mAP50:95.
A displayed confidence of 0.80 is a filtering threshold, not 80% accuracy.

## Why this version should improve

Local profiling found that BDD100K traffic lights and signs are overwhelmingly
tiny: 97.7% and 89.5% of their boxes occupy less than 0.25% of the image.
The completed main manifest had 1,200 scenes but only 215 with buses, 376 with
trucks, and 441 with pedestrians. This version uses 4,800 main scenes with
explicit class minimums, then 3,200 more strongly balanced refinement scenes.
The 576 px main stage reduces pixel work enough to process this broader sample;
704 px refinement restores detail for distant pedestrians, lights, and signs.

The workflow follows transfer-learning and performance recommendations from
[Ultralytics training settings](https://docs.ultralytics.com/modes/train/),
[YOLO11 model scaling](https://docs.ultralytics.com/models/yolo11/), and the
[BDD100K paper](https://arxiv.org/abs/1805.04687). The linked road-segmentation
project reinforces brightness/contrast augmentation and pretrained encoders,
but semantic segmentation is a different output task, so it is not substituted
for box detection here.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_detection.yolo_training_utils import (
    label_path_for_image,
    prepare_fast_data_files,
    read_image_manifest,
    stage_fast_data_files,
    stage_images_and_labels,
    uniform_sample,
    write_image_manifest,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", __import__("ultralytics").__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

EXPECTED_VENV = Path(r"C:\tf214_hw2\Scripts\python.exe")
if EXPECTED_VENV.exists() and Path(sys.executable).resolve() != EXPECTED_VENV.resolve():
    raise RuntimeError(
        "Select the Jupyter kernel 'Python (tf214_hw2)' and restart the notebook. "
        f"Current interpreter: {sys.executable}"
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [ ]:
# Use "smoke" once after changing code. Use "rtx3050_more_data" for the real run.
RUN_MODE = "rtx3050_more_data"
BASE_RUN_TAG = "yolo11s_576_class_coverage_v3"
RESUME_TRAINING = True
RUN_REFINEMENT = True

PROFILES = {
    "smoke": {
        "model": "yolo11s.pt",
        "main_count": 128,
        "refine_count": 128,
        "epoch_val_count": 64,
        "candidate_scan_count": 512,
        "main_epochs": 2,
        "refine_epochs": 1,
        "imgsz": 512,
        "refine_imgsz": 576,
        "batch": 8,
        "refine_batch": 8,
        "patience": 3,
        "final_eval_count": 128,
        "main_class_minimums": [100, 20, 30, 40, 60, 90],
        "refine_class_minimums": [100, 25, 35, 45, 70, 95],
    },
    "rtx3050_more_data": {
        "model": "yolo11s.pt",
        "main_count": 4800,
        "refine_count": 3200,
        "epoch_val_count": 600,
        "candidate_scan_count": 40000,
        "main_epochs": 32,
        "refine_epochs": 10,
        "imgsz": 576,
        "refine_imgsz": 704,
        "batch": 16,
        "refine_batch": 12,
        "patience": 8,
        "final_eval_count": 2000,
        "main_class_minimums": [4500, 1000, 1500, 1800, 2800, 4000],
        "refine_class_minimums": [3000, 800, 1100, 1300, 2000, 2800],
    },
    "overnight_accuracy": {
        "model": "yolo11s.pt",
        "main_count": 6400,
        "refine_count": 4800,
        "epoch_val_count": 800,
        "candidate_scan_count": 60000,
        "main_epochs": 36,
        "refine_epochs": 12,
        "imgsz": 576,
        "refine_imgsz": 704,
        "batch": 16,
        "refine_batch": 12,
        "patience": 10,
        "final_eval_count": 2000,
        "main_class_minimums": [6000, 1400, 2200, 2600, 3800, 5400],
        "refine_class_minimums": [4500, 1200, 1700, 2000, 3000, 4200],
    },
}
CFG = PROFILES[RUN_MODE]
RUN_TAG = f"{BASE_RUN_TAG}_{RUN_MODE}"

if RUN_MODE != "smoke" and not torch.cuda.is_available():
    raise RuntimeError(
        "The real profiles require CUDA. The prior notebook accidentally used "
        "CPU-only PyTorch; repair/select the tf214_hw2 kernel before training."
    )

DEVICE = 0 if torch.cuda.is_available() else "cpu"
# RAM-cached datasets are fastest with no Windows worker respawn/copy overhead.
WORKERS = 0
DATA_YAML = PROJECT_ROOT / "data" / "bdd100k_yolo" / "data.yaml"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "yolo_accuracy"
MANIFEST_DIR = RUN_ROOT / "manifests" / RUN_TAG
PRIOR_BEST = PROJECT_ROOT / "outputs" / "yolo_final" / "bdd100k_yolo11s_best.pt"
START_MODEL = str(PRIOR_BEST) if PRIOR_BEST.exists() else CFG["model"]
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "yolo_more_data"
FAST_CACHE_BASE = Path(
    os.environ.get("BDD100K_FAST_CACHE")
    or Path(os.environ.get("LOCALAPPDATA", PROJECT_ROOT / "tmp"))
    / "bdd100k_road_detection_cache"
)
LOCAL_CACHE_ROOT = FAST_CACHE_BASE / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Missing {DATA_YAML}. Run scripts/prepare_bdd100k.ps1 with the real "
        "BDD100K folder first."
    )
print(pd.Series(CFG, name=RUN_MODE))

In [ ]:
data_files = prepare_fast_data_files(
    data_yaml=DATA_YAML,
    output_dir=MANIFEST_DIR,
    run_root=RUN_ROOT,
    preferred_manifest_tag=RUN_TAG,
    main_count=CFG["main_count"],
    refine_count=CFG["refine_count"],
    validation_count=CFG["epoch_val_count"],
    seed=SEED,
    candidate_scan_count=CFG["candidate_scan_count"],
    scan_workers=12,
    main_class_minimums=CFG["main_class_minimums"],
    refine_class_minimums=CFG["refine_class_minimums"],
)
staging_started = time.perf_counter()
data_files = stage_fast_data_files(
    data_files,
    cache_root=LOCAL_CACHE_ROOT,
    source_data_yaml=DATA_YAML,
    workers=12,
)
print(f"One-time/reusable local staging: {time.perf_counter() - staging_started:.1f}s")
print("Sampling source:", data_files.source)
print("Main images:", sum(1 for _ in data_files.main_manifest.open()))
print("Refine images:", sum(1 for _ in data_files.refine_manifest.open()))
print("Epoch validation images:", sum(1 for _ in data_files.validation_manifest.open()))
print("Main YAML:", data_files.main_yaml)


def class_coverage_table(manifest):
    image_counts = np.zeros(6, dtype=int)
    box_counts = np.zeros(6, dtype=int)
    images = read_image_manifest(manifest)
    for image_path in images:
        seen = set()
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            continue
        for line in label_path.read_text(encoding="utf-8").splitlines():
            fields = line.split()
            if len(fields) < 5:
                continue
            class_id = int(float(fields[0]))
            box_counts[class_id] += 1
            seen.add(class_id)
        for class_id in seen:
            image_counts[class_id] += 1
    return pd.DataFrame({
        "class": ["car", "bus", "truck", "pedestrian", "traffic light", "traffic sign"],
        "images": image_counts,
        "boxes": box_counts,
        "image_percent": 100.0 * image_counts / len(images),
    })

print("Main class coverage")
display(class_coverage_table(data_files.main_manifest))
print("Refinement class coverage")
display(class_coverage_table(data_files.refine_manifest))

## Main transfer-learning stage

The main stage trains one continuous optimizer schedule. This avoids the
previous notebook's short frozen/unfrozen restarts, which reset momentum and
produced a worse second stage. Rectangular batches avoid square-image padding
and mosaic shrinkage; scale, color, and horizontal-flip augmentation still cover
BDD100K lighting, weather, and viewpoint variation. Moderate inverse-frequency
classification weights give buses and trucks more useful gradient influence.

A one-time parallel staging cell hydrates the selected files into a separate
local cache. The first CUDA epoch includes kernel and RAM-cache warm-up. The
larger 4,800-image epochs are expected to take longer than the completed run,
but only 32 are scheduled; the total default profile is intended for overnight
training on the 4 GB RTX 3050.

In [ ]:
MAIN_NAME = f"{RUN_TAG}_main"
MAIN_DIR = RUN_ROOT / MAIN_NAME
MAIN_LAST = MAIN_DIR / "weights" / "last.pt"

if RESUME_TRAINING and MAIN_LAST.exists():
    print("Resuming:", MAIN_LAST)
    main_model = YOLO(str(MAIN_LAST))
    main_model.train(resume=True)
else:
    print("Starting from:", START_MODEL)
    main_model = YOLO(START_MODEL)
    main_model.train(
        data=str(data_files.main_yaml),
        epochs=CFG["main_epochs"],
        imgsz=CFG["imgsz"],
        batch=CFG["batch"],
        device=DEVICE,
        workers=WORKERS,
        cache="ram" if torch.cuda.is_available() else False,
        rect=True,
        amp=torch.cuda.is_available(),
        optimizer="AdamW",
        lr0=0.0008,
        lrf=0.05,
        cos_lr=True,
        warmup_epochs=1.0,
        weight_decay=0.0005,
        box=7.5,
        cls=0.85,
        cls_pw=0.25,
        dfl=1.5,
        hsv_h=0.015,
        hsv_s=0.55,
        hsv_v=0.35,
        degrees=2.0,
        translate=0.08,
        scale=0.25,
        shear=0.0,
        perspective=0.0002,
        fliplr=0.5,
        flipud=0.0,
        mosaic=0.0,
        mixup=0.0,
        close_mosaic=0,
        # Fixed shapes avoid repeated cuDNN autotuning on this 4 GB GPU.
        patience=CFG["patience"],
        max_det=300,
        plots=True,
        deterministic=False,
        seed=SEED,
        project=str(RUN_ROOT),
        name=MAIN_NAME,
        exist_ok=True,
        verbose=True,
    )

MAIN_BEST = MAIN_DIR / "weights" / "best.pt"
if not MAIN_BEST.exists():
    raise FileNotFoundError(f"Training did not create {MAIN_BEST}")
print("Main candidate:", MAIN_BEST)
del main_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Small-object refinement

Refinement starts from the main best checkpoint, uses the stronger
small-object-biased manifest, increases resolution to 704 px, lowers the
learning rate, and mostly disables mosaic. It is deliberately short.
The next section evaluates both checkpoints on the exact same images and
rejects refinement if it regresses.

In [ ]:
REFINE_NAME = f"{RUN_TAG}_refine"
REFINE_DIR = RUN_ROOT / REFINE_NAME
REFINE_LAST = REFINE_DIR / "weights" / "last.pt"
REFINE_BEST = REFINE_DIR / "weights" / "best.pt"

if RUN_REFINEMENT:
    if RESUME_TRAINING and REFINE_LAST.exists():
        print("Resuming:", REFINE_LAST)
        refine_model = YOLO(str(REFINE_LAST))
        refine_model.train(resume=True)
    else:
        refine_model = YOLO(str(MAIN_BEST))
        refine_model.train(
            data=str(data_files.refine_yaml),
            epochs=CFG["refine_epochs"],
            imgsz=CFG["refine_imgsz"],
            batch=CFG["refine_batch"],
            device=DEVICE,
            workers=WORKERS,
            cache="ram" if torch.cuda.is_available() else False,
            rect=True,
            amp=torch.cuda.is_available(),
            optimizer="AdamW",
            lr0=0.00030,
            lrf=0.10,
            cos_lr=True,
            warmup_epochs=1.0,
            weight_decay=0.0005,
            box=8.0,
            cls=0.90,
            cls_pw=0.35,
            dfl=1.5,
            hsv_h=0.012,
            hsv_s=0.40,
            hsv_v=0.28,
            degrees=1.0,
            translate=0.05,
            scale=0.20,
            fliplr=0.5,
            mosaic=0.0,
            mixup=0.0,
            close_mosaic=0,
            patience=max(4, CFG["refine_epochs"] // 2),
            max_det=300,
            plots=True,
            deterministic=False,
            seed=SEED + 1,
            project=str(RUN_ROOT),
            name=REFINE_NAME,
            exist_ok=True,
            verbose=True,
        )
    if not REFINE_BEST.exists():
        raise FileNotFoundError(f"Refinement did not create {REFINE_BEST}")
    del refine_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## No-regression model selection

Candidates are compared on one untouched fixed validation manifest. Selection
emphasizes mAP50 while still rewarding F1 and localization quality. The exact
checkpoint from the completed 01 run is included, so the larger run must
actually beat the result you already like rather than merely finish training.

In [ ]:
def summarize_yolo(metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    quality = 0.55 * map50 + 0.30 * f1 + 0.15 * map50_95
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "map50": map50,
        "map50_95": map50_95,
        "quality": quality,
    }

candidates = {"main": MAIN_BEST}
if RUN_REFINEMENT and REFINE_BEST.exists():
    candidates["refine"] = REFINE_BEST
if PRIOR_BEST.exists():
    candidates["completed_01_baseline"] = PRIOR_BEST
OLD_BASELINE = (
    PROJECT_ROOT / "runs" / "notebooks" / "yolo"
    / "bdd100k_cpu_quick_finetuned" / "weights" / "best.pt"
)
if OLD_BASELINE.exists():
    candidates["old_cpu_baseline"] = OLD_BASELINE

candidate_rows = []
candidate_metrics = {}
for name, weights in candidates.items():
    print(f"Evaluating {name}: {weights}")
    metrics = YOLO(str(weights)).val(
        data=str(data_files.main_yaml),
        imgsz=CFG["refine_imgsz"],
        batch=CFG["refine_batch"],
        device=DEVICE,
        workers=WORKERS,
        conf=0.001,
        iou=0.70,
        max_det=300,
        plots=False,
        verbose=False,
    )
    row = {"candidate": name, "weights": str(weights), **summarize_yolo(metrics)}
    candidate_rows.append(row)
    candidate_metrics[name] = metrics

candidate_table = pd.DataFrame(candidate_rows).sort_values("quality", ascending=False)
display(candidate_table)
winner_name = str(candidate_table.iloc[0]["candidate"])
winner_path = Path(candidate_table.iloc[0]["weights"])
winner_metrics = candidate_metrics[winner_name]
FINAL_WEIGHTS = OUTPUT_DIR / "bdd100k_yolo11s_more_data_best.pt"
shutil.copy2(winner_path, FINAL_WEIGHTS)
candidate_table.to_csv(OUTPUT_DIR / "candidate_comparison.csv", index=False)
print("Selected:", winner_name)
print("Deployment weights:", FINAL_WEIGHTS)

In [ ]:
# Build reproducible, bounded final validation and test manifests.
base_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(base_config["path"])
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

def split_dir(split_name):
    value = Path(base_config[split_name])
    return value if value.is_absolute() else dataset_root / value

final_val_images = uniform_sample(
    sorted(split_dir("val").glob("*.jpg")),
    CFG["final_eval_count"],
    SEED + 20,
)
final_test_dir = split_dir("test") if "test" in base_config else None
if final_test_dir is None or not final_test_dir.exists():
    raise FileNotFoundError("Converted data.yaml must contain the held-out local test split.")
final_test_images = uniform_sample(
    sorted(final_test_dir.glob("*.jpg")),
    CFG["final_eval_count"],
    SEED + 21,
)
final_val_images = stage_images_and_labels(
    final_val_images, LOCAL_CACHE_ROOT, "val", workers=12
)
final_test_images = stage_images_and_labels(
    final_test_images, LOCAL_CACHE_ROOT, "test", workers=12
)
final_val_manifest = write_image_manifest(MANIFEST_DIR / "final_val.txt", final_val_images)
final_test_manifest = write_image_manifest(MANIFEST_DIR / "final_test.txt", final_test_images)

final_config = dict(base_config)
final_config["path"] = dataset_root.as_posix()
final_config["val"] = final_val_manifest.resolve().as_posix()
final_config["test"] = final_test_manifest.resolve().as_posix()
FINAL_DATA_YAML = MANIFEST_DIR / "final_eval.yaml"
FINAL_DATA_YAML.write_text(yaml.safe_dump(final_config, sort_keys=False), encoding="utf-8")

final_model = YOLO(str(FINAL_WEIGHTS))
final_val_metrics = final_model.val(
    data=str(FINAL_DATA_YAML),
    split="val",
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=True,
    verbose=False,
)
final_test_metrics = final_model.val(
    data=str(FINAL_DATA_YAML),
    split="test",
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=False,
    verbose=False,
)
final_table = pd.DataFrame(
    [
        {"split": "validation", **summarize_yolo(final_val_metrics)},
        {"split": "test", **summarize_yolo(final_test_metrics)},
    ]
)
display(final_table)
final_table.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)

## Calibrate balanced and high-precision live thresholds

The balanced profile chooses each class's best validation F1. The
high-precision profile chooses the lowest threshold that reaches at least
80% precision when the validation curve supports it, with a hard displayed
confidence floor of 0.70. This can produce 0.70-0.80 confidence labels while
remaining honest about the resulting recall.

In [ ]:
def calibrate_thresholds(metrics, target_precision=0.80, confidence_floor=0.70):
    box = metrics.box
    px = np.asarray(box.px)
    p_curve = np.asarray(box.p_curve)
    r_curve = np.asarray(box.r_curve)
    f1_curve = np.asarray(box.f1_curve)
    names = metrics.names
    balanced = {}
    high_precision = {}
    rows = []
    for class_id in range(len(names)):
        best_f1_index = int(np.nanargmax(f1_curve[class_id]))
        balanced_threshold = float(px[best_f1_index])
        valid = np.flatnonzero(
            (p_curve[class_id] >= target_precision)
            & (px >= confidence_floor)
        )
        if len(valid):
            strict_index = int(valid[np.nanargmax(r_curve[class_id, valid])])
        else:
            strict_index = int(np.argmin(np.abs(px - confidence_floor)))
        strict_threshold = max(confidence_floor, float(px[strict_index]))
        balanced[str(class_id)] = round(max(0.05, balanced_threshold), 3)
        high_precision[str(class_id)] = round(strict_threshold, 3)
        rows.append(
            {
                "class_id": class_id,
                "class": names[class_id],
                "balanced_threshold": balanced[str(class_id)],
                "balanced_f1": float(f1_curve[class_id, best_f1_index]),
                "strict_threshold": high_precision[str(class_id)],
                "strict_precision": float(p_curve[class_id, strict_index]),
                "strict_recall": float(r_curve[class_id, strict_index]),
                "target_precision_met": bool(p_curve[class_id, strict_index] >= target_precision),
            }
        )
    return balanced, high_precision, pd.DataFrame(rows)

balanced_thresholds, strict_thresholds, threshold_table = calibrate_thresholds(
    final_val_metrics
)
display(threshold_table)
threshold_table.to_csv(OUTPUT_DIR / "threshold_calibration.csv", index=False)

deployment_config = {
    "backend": "yolo",
    "weights": FINAL_WEIGHTS.name,
    "imgsz": CFG["refine_imgsz"],
    "max_det": 300,
    "default_threshold_profile": "balanced",
    "threshold_profiles": {
        "balanced": {
            "inference_confidence": 0.05,
            "class_thresholds": balanced_thresholds,
            "meaning": "Per-class validation F1 optimum",
        },
        "high_precision_80": {
            "inference_confidence": 0.70,
            "class_thresholds": strict_thresholds,
            "target_validation_precision": 0.80,
            "meaning": "At least 0.70 displayed confidence; class target is 0.80 precision where supported",
        },
    },
    "selection": winner_name,
    "validation_metrics": summarize_yolo(final_val_metrics),
    "test_metrics": summarize_yolo(final_test_metrics),
}
DEPLOYMENT_CONFIG = OUTPUT_DIR / "deployment_config.json"
DEPLOYMENT_CONFIG.write_text(json.dumps(deployment_config, indent=2), encoding="utf-8")
print("Deployment config:", DEPLOYMENT_CONFIG)

In [ ]:
# Reload after validation so the predictor owns ordinary, mutable tensors.
benchmark_model = YOLO(str(FINAL_WEIGHTS))
# Warm GPU, measure end-to-end detector throughput, and show representative output.
benchmark_images = [str(path) for path in final_val_images[:100]]
_ = benchmark_model.predict(
    benchmark_images[:8],
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    quantize=16 if torch.cuda.is_available() else None,
    conf=0.25,
    verbose=False,
)
if torch.cuda.is_available():
    torch.cuda.synchronize()
started = time.perf_counter()
benchmark_results = benchmark_model.predict(
    benchmark_images,
    imgsz=CFG["refine_imgsz"],
    batch=CFG["refine_batch"],
    device=DEVICE,
    quantize=16 if torch.cuda.is_available() else None,
    conf=0.25,
    verbose=False,
)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - started
print(f"End-to-end throughput: {len(benchmark_images) / elapsed:.2f} images/s")
print(f"Latency: {1000 * elapsed / len(benchmark_images):.1f} ms/image")

figure, axes = plt.subplots(2, 2, figsize=(16, 9))
for axis, result in zip(axes.flat, benchmark_results[:4]):
    axis.imshow(result.plot()[..., ::-1])
    axis.axis("off")
plt.tight_layout()
plt.show()

## Realtime command

```powershell
python -m road_detection.realtime_detect `
  --backend yolo `
  --weights outputs\yolo_more_data\bdd100k_yolo11s_more_data_best.pt `
  --config outputs\yolo_more_data\deployment_config.json `
  --threshold-profile high_precision_80 `
  --source 0 `
  --device 0
```

Use `balanced` for the best precision/recall tradeoff. Use
`high_precision_80` when cleaner, 0.70+ confidence labels matter more than
recall. Keep the candidate comparison and final tables with the project
report; they show whether the requested target was measured rather than
assumed.